<a href="https://colab.research.google.com/github/Matheusfbio/AI-Labs/blob/main/huggingface_ai_laboratory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratório: Chatbot Interativo com Qwen 2.5-0.5B-Instruct

Este laboratório demonstra como configurar e interagir com o modelo de linguagem **Qwen2.5-0.5B-Instruct** localmente no ambiente do Google Colab usando a GPU (Tesla T4 ou equivalente) para inferência rápida.

## 🚀 Tecnologias Utilizadas

- **Python** (versão 3.10+)
- **PyTorch** com suporte a CUDA (GPU)
- **Hugging Face Transformers**: Para carregar o modelo de linguagem e o tokenizador.
- **Qwen2.5-0.5B-Instruct**: Um modelo compacto e altamente otimizado capaz de seguir instruções e manter conversas em português.

## 📂 Estrutura do Notebook

1. **Instalação de Dependências**: Instala as bibliotecas modernas do ecossistema Hugging Face (`transformers`, `accelerate`, etc.).
2. **Verificação de Hardware**: Valida se a execução está sendo realizada em uma GPU ativa para melhor desempenho.
3. **Carregamento do Modelo**: Baixa e carrega o tokenizador e os pesos do modelo diretamente do Hugging Face Hub.
4. **Chat Interativo em Loop**: Implementa um loop contínuo de conversação (`while True`) onde você pode enviar suas próprias perguntas diretamente pela barra de entrada do console até decidir digitar `sair`.

## 🛠️ Como Usar

1. Execute a primeira célula para instalar as dependências.
2. Execute a célula de validação de ambiente para garantir que a GPU está disponível.
3. Inicialize o modelo carregando os pesos.
4. Execute a última célula para iniciar o chat em loop. Digite sua pergunta e aperte `Enter` para interagir com a IA.
5. Para finalizar o chat, basta digitar `sair`, `parar` ou `exit` no campo de texto.

In [1]:

!pip -q install -U \
    transformers \
    accelerate \
    sentence-transformers \
    datasets \
    evaluate \
    huggingface_hub \
    langchain \
    langchain-core \
    langchain-huggingface


In [1]:

import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Memória GPU:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("Executand
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)

print("Modelo carregado!")
print("Dispositivo:", model.device)
o em CPU")


Python: 3.13.16
PyTorch: 2.14.1+cu130
CUDA disponível: True
GPU: Tesla T4
Memória GPU: 14.56 GB


In [2]:

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)

print("Modelo carregado!")
print("Dispositivo:", model.device)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo carregado!
Dispositivo: cuda:0


In [ ]:
import torch

# Função que envia a mensagem para o modelo e retorna a resposta gerada
def chat(message, max_tokens=300):
    # Define o histórico/formato da conversa seguindo os papéis (roles) suportados pelo modelo
    messages = [
        {
            "role": "system",
            "content": "Você é um assistente especializado em tecnologia."
        },
        {
            "role": "user",
            "content": message
        }
    ]

    # Aplica o template de chat específico do modelo (adicionando os tokens especiais de início/fim de turno)
    # E converte tudo para tensores do PyTorch (.to(model.device) move os dados para a GPU/CPU onde o modelo está)
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # Desabilita o cálculo de gradientes para economizar memória e acelerar a inferência
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,        # Permite amostragem probabilística para respostas mais criativas/variadas
            temperature=0.7,       # Controla a criatividade (valores menores são mais deterministas)
            top_p=0.9,             # Filtra as palavras mais prováveis que somam 90% de probabilidade
            pad_token_id=tokenizer.eos_token_id
        )

    # Remove os tokens de entrada da resposta gerada para obter apenas o que o modelo respondeu
    generated_tokens = outputs[
        :, inputs["input_ids"].shape[1]:
    ]

    # Decodifica os tokens numéricos gerados de volta para texto legível em português
    response = tokenizer.decode(
        generated_tokens[0],
        skip_special_tokens=True
    )

    return response.strip()

# Loop contínuo para manter a conversa ativa
print("Iniciando o Chat contínuo. Digite 'sair', 'exit' ou 'parar' para encerrar.\n")

while True:
    pergunta_usuario = input("Você: ")

    # Verifica se o usuário deseja encerrar o loop
    if pergunta_usuario.strip().lower() in ["sair", "exit", "parar"]:
        print("\nConversa encerrada. Até a próxima!")
        break

    if not pergunta_usuario.strip():
        continue

    print("\nProcessando sua resposta...\n")
    response = chat(pergunta_usuario)

    print(f"Modelo: {response}")
    print("-" * 50 + "\n")

Iniciando o Chat contínuo. Digite 'sair', 'exit' ou 'parar' para encerrar.

Você: ola tudo bem?

Processando sua resposta...

Modelo: Olá! Estou bem, obrigado por perguntar. Como posso ajudar você hoje?
--------------------------------------------------

Você: Que modelo voce é ?

Processando sua resposta...

Modelo: Eu sou um modelo de inteligência artificial desenvolvido pela Anthropic para responder perguntas e fornecer informações baseadas em dados.
--------------------------------------------------

Você: tem certeza ?

Processando sua resposta...

Modelo: Sim, eu posso ser útil! Qualquer outra coisa você precisa saber?
--------------------------------------------------

Você: quais os beneficios de usar a linguagem de programação elixir com o framework phoenix ?

Processando sua resposta...

Modelo: O Elixir e Phoenix são frameworks populares para desenvolvimento web, mas também oferecem algumas vantagens adicionais:

1. Flexibilidade: Phoenix incorpora várias características do 